# Navitas — Model Güçlendirme (v3) · Colab

**Amaç:** mask mAP50'yi **0,60 takım hedefinin üstüne** çıkarmak (hedef ≥ 0,65).
v2 = 0,595 (blocked 0,322 yüzünden hedefin altında). v3 bu boşluğu kapatır.

**İki bölüm:**
- **BÖLÜM 1 — Eğitim + Güçlendirme:** RescueNet indir → blocked dengele → YOLOv8**s**-seg
  ile güçlendirilmiş reçeteyle eğit → ağırlıkları kaydet/indir.
- **BÖLÜM 2 — Test + Rapor:** v3'ü ölç (sınıf bazlı) → v1/v2/v3 kıyas → zip indir → Claude DDR v4.3'e işler.

**Neden güçlü olacak (kanıta dayalı):**
1. Model nano→**small** (kapasite 3×) · 2. `degrees` 45→15 (v2'nin blocked gerilemesini geri al)
· 3. `copy_paste` 0.15→0.30 (azınlık boost) · 4. blocked **oversampling** · 5. 200 epoch.

**Güvenlik ağı:** v1 (0,619) zaten 0,60'ı geçiyor → v3 tamamen yukarı yönlü risk.

**Başlamadan:** `Çalışma zamanı → türü değiştir → **T4 GPU**` (eğitim GPU ŞART).
Hücreleri sırayla çalıştır, her birinde `✓ ... TAMAM` gör.

---
# BÖLÜM 1 — EĞİTİM + GÜÇLENDİRME

## 1.1 — Ortam + log
Tüm çıktı ekranda görünür; ayrıca `konsol.log`'a yazılır.

In [ ]:
import os, sys, json, subprocess, datetime, platform

CALISMA = "/content/navitas_v3"
os.makedirs(CALISMA, exist_ok=True)
LOG = f"{CALISMA}/konsol.log"

def logla(*p):
    s = " ".join(str(x) for x in p)
    print(s)
    with open(LOG, "a", encoding="utf-8") as f:
        f.write(s + "\n")

def komut(cmd, gizle=None):
    logla(f"\n$ {cmd.replace(gizle,'***') if gizle else cmd}")
    pr = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, errors="replace")
    for satir in pr.stdout:
        logla(satir.rstrip("\n").split("\r")[-1])
    pr.wait()
    logla(f"[çıkış: {pr.returncode}]")
    return pr.returncode

logla("="*60); logla("NAVITAS v3 GÜÇLENDİRME —", datetime.datetime.now().isoformat()); logla("="*60)
komut("pip install -q ultralytics datasets pillow tqdm opencv-python-headless")
import torch, ultralytics
ortam = {"tarih": datetime.datetime.now().isoformat(), "python": platform.python_version(),
         "torch": torch.__version__, "ultralytics": ultralytics.__version__,
         "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "yok (CPU)"}
json.dump(ortam, open(f"{CALISMA}/ortam.json","w"), indent=2, ensure_ascii=False)
logla("Ortam:", json.dumps(ortam, ensure_ascii=False))
logla("\n✓ 1.1 TAMAM")

## 1.2 — Repo + v3 dosyaları doğrula
Token'ı yapıştır (repo gizli). Klon sonrası v3 scriptlerinin geldiğini kontrol eder = son push ulaştı mı.

In [ ]:
from getpass import getpass
if not os.path.isdir("/content/navitas/.git"):
    token = getpass("GitHub token: ").strip()
    url = f"https://{token}@github.com/navitas26/navitas.git" if token else "https://github.com/navitas26/navitas.git"
    if komut(f"git clone --depth 1 {url} /content/navitas", gizle=token or None) != 0:
        raise SystemExit("Klon BAŞARISIZ — token/internet kontrol")
else:
    logla("Repo zaten var.")

gerekli = ["/content/navitas/codes/train_seg_v3.py",
           "/content/navitas/dataset/oversample_blocked.py",
           "/content/navitas/dataset/download_and_convert.py"]
eksik = [f for f in gerekli if not os.path.isfile(f)]
if eksik:
    logla("❌ v3 dosyaları eksik (son push ulaşmamış?):", eksik)
    raise SystemExit("Repo güncel değil — Claude'a söyle, önce push gerekli")
logla("✅ v3 dosyaları repoda — son push ulaşmış")
logla("\n✓ 1.2 TAMAM")

## 1.3 — GPU ZORUNLU
YOLOv8s 200 epoch CPU'da günler sürer. GPU yoksa burada durur.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit("❌ GPU YOK! Çalışma zamanı → türü değiştir → T4 GPU seç, sonra baştan çalıştır.")
logla(f"✅ GPU: {torch.cuda.get_device_name(0)}")
logla("\n✓ 1.3 TAMAM")

## 1.4 — Veri: RescueNet indir

**Not:** FloodNet BİLEREK atlanıyor — mevcut indirici Track-2 (VQA, maskesiz) çekiyordu; gerçek katkısı hiç olmadı (v1/v2 de fiilen RescueNet-only'di). Düzeltme ayrı bir görev; v3 güvenilir şekilde RescueNet + dengeleme ile hedefi geçer.

In [ ]:
os.chdir("/content/navitas/dataset")
if komut("python3 download_and_convert.py") != 0:
    raise SystemExit("RescueNet indirme başarısız")
import glob
tr = len(glob.glob("/content/navitas/dataset/rescuenet_yolo/images/train/*"))
va = len(glob.glob("/content/navitas/dataset/rescuenet_yolo/images/val/*"))
logla(f"RescueNet: train={tr} val={va}")
assert tr > 0 and va > 0, "Veri boş — indirme gerçekten bitti mi?"
logla("\n✓ 1.4 TAMAM")

## 1.5 — Ayar (RescueNet-only) + blocked dengeleme
Val = 294 görüntülük AYNI set (v1/v2 ile kıyaslanabilir). Sonra blocked oversampling (train'e, +2 kopya).

In [ ]:
yaml_icerik = """train: /content/navitas/dataset/rescuenet_yolo/images/train
val:   /content/navitas/dataset/rescuenet_yolo/images/val
test:  /content/navitas/dataset/rescuenet_yolo/images/test
nc: 2
names:
  0: traversable
  1: blocked
"""
VERI_YAML = "/content/navitas/dataset/v3_data.yaml"
open(VERI_YAML, "w").write(yaml_icerik)
logla(yaml_icerik)

# blocked oversampling (idempotent; kapatmak için --copies 0)
komut("python3 /content/navitas/dataset/oversample_blocked.py "
      "--dir /content/navitas/dataset/rescuenet_yolo --split train --copies 2")
logla("\n✓ 1.5 TAMAM")

## 1.6 — EĞİT (YOLOv8s-seg, 200 epoch · ~1–2 saat ☕)
Güçlendirilmiş reçete `codes/train_seg_v3.py` içinde (sürüm kontrollü). Sekmeyi kapatma.

In [ ]:
os.chdir("/content/navitas")
rc = komut("python3 codes/train_seg_v3.py --device 0 --model yolov8s-seg.pt "
           f"--epochs 200 --batch 16 --imgsz 512 --data {VERI_YAML} --name navitas_seg_v3")
if rc != 0:
    logla("⚠ Eğitim hata kodu döndü — yine de Bölüm 2 eldeki en iyi ağırlıkla çalışabilir.")
logla("\n✓ 1.6 TAMAM (eğitim bitti)")

## 1.7 — Ağırlıkları kaydet + indir (+ opsiyonel push)
best.pt → models/, SHA, zip indir. Push token izin verirse dener (yoksa indirme yeter).

In [ ]:
import hashlib, shutil, glob
adaylar = glob.glob("/content/navitas/codes/runs/segment/navitas_seg_v3/weights/best.pt") + \
          glob.glob("/content/navitas/**/navitas_seg_v3/weights/best.pt", recursive=True)
if not adaylar:
    raise SystemExit("best.pt bulunamadı — eğitim gerçekten bitti mi?")
BEST = adaylar[0]
HEDEF = "/content/navitas/models/navitas_seg_v3_best.pt"
os.makedirs("/content/navitas/models", exist_ok=True)
shutil.copy2(BEST, HEDEF)
sha = hashlib.sha256(open(HEDEF,"rb").read()).hexdigest()
logla(f"best.pt: {BEST}\nSHA-256: {sha}  ·  {os.path.getsize(HEDEF)/1e6:.1f} MB")
json.dump({"sha256": sha, "kaynak": BEST}, open(f"{CALISMA}/v3_model_kimlik.json","w"), indent=2)

# eğitim çıktılarını + ağırlığı zip'le, indir
runzip = f"/content/navitas_v3_weights_{datetime.datetime.now():%Y%m%d_%H%M}"
rundir = os.path.dirname(os.path.dirname(BEST))
shutil.copy2(HEDEF, f"{rundir}/navitas_seg_v3_best.pt")
shutil.make_archive(runzip, "zip", rundir)
logla("Ağırlık+eğitim zip:", runzip + ".zip")

# opsiyonel: GitHub'a push (izin varsa)
try:
    os.chdir("/content/navitas")
    komut('git config user.name "trs-1342"')
    komut('git config user.email "socialmediaoftrs1342@gmail.com"')
    open("/content/navitas/push_isareti_v3.txt","w").write("NAVITAS-V3-EGITILDI")
    komut("git add -f models/navitas_seg_v3_best.pt push_isareti_v3.txt")
    komut('git commit -m "model: navitas_seg_v3 (yolov8s-seg, guclendirilmis rece + oversampling)"')
    if komut("git push origin master") == 0:
        logla("✅ v3 modeli GitHub'a pushlandı")
    else:
        logla("ℹ Push olmadı (token yazma izni yok olabilir) — sorun değil, zip indirildi.")
except Exception as e:
    logla("ℹ Push denemesi atlandı:", e)

from google.colab import files
files.download(runzip + ".zip")
logla("\n✓ 1.7 TAMAM — BÖLÜM 1 bitti. Şimdi BÖLÜM 2'ye geç.")

---
# BÖLÜM 2 — TEST + RAPOR

## 2.1 — v3'ü ölç (sınıf bazlı, confusion + PR grafikleri)
Aynı 294 görüntülük RescueNet val setinde — v1/v2 ile birebir kıyaslanabilir.

In [ ]:
from ultralytics import YOLO
import logging
fh = logging.FileHandler(LOG); fh.setFormatter(logging.Formatter("%(message)s"))
logging.getLogger("ultralytics").addHandler(fh)

model = YOLO(HEDEF)
r = model.val(data=VERI_YAML, imgsz=512, split="val", plots=True,
              project=CALISMA, name="v3_val", exist_ok=True)
v3 = {"genel": {"mask_mAP50": round(r.seg.map50,4), "mask_mAP50_95": round(r.seg.map,4),
                "box_mAP50": round(r.box.map50,4), "mask_precision": round(r.seg.mp,4),
                "mask_recall": round(r.seg.mr,4)},
      "sinif": {r.names[int(c)]: {"mask_mAP50": round(float(r.seg.ap50[i]),4),
                                  "mask_mAP50_95": round(float(r.seg.ap[i]),4)}
                for i,c in enumerate(r.ap_class_index)},
      "hiz_ms": r.speed}
json.dump(v3, open(f"{CALISMA}/v3_val_sonuclari.json","w"), indent=2, ensure_ascii=False)
logla("\nv3 SONUÇ:", json.dumps(v3, ensure_ascii=False))
logla("\n✓ 2.1 TAMAM")

## 2.2 — Kıyas: v1 / v2 / v3 (aynı 294 val seti)
v1, v2 kayıtlı referans; v3 taze ölçüm.

In [ ]:
REF = {  # aynı RescueNet val setinde daha önce ölçülen değerler
  "v1 (rescuenet_seg_v1)": {"genel": 0.619, "traversable": 0.874, "blocked": 0.363},
  "v2 (navitas_seg_v2)":   {"genel": 0.595, "traversable": 0.868, "blocked": 0.322},
}
v3row = {"genel": v3["genel"]["mask_mAP50"],
         "traversable": v3["sinif"].get("traversable",{}).get("mask_mAP50"),
         "blocked": v3["sinif"].get("blocked",{}).get("mask_mAP50")}
tablo = dict(REF); tablo["v3 (navitas_seg_v3) ⭐"] = v3row

logla("\n| Model | mask mAP50 (genel) | traversable | blocked |")
logla("|---|---|---|---|")
for ad, d in tablo.items():
    logla(f"| {ad} | {d['genel']} | {d['traversable']} | {d['blocked']} |")

json.dump(tablo, open(f"{CALISMA}/kiyas.json","w"), indent=2, ensure_ascii=False)
g = v3row["genel"]
logla(f"\n>>> HEDEF mask mAP50 ≥ 0,60 → v3 = {g} → " + ("GEÇTİ ✅✅✅" if g>=0.60 else "geçmedi ❌ (m-model / veri genişletme dene)"))
if g >= 0.60:
    onceki = max(REF['v1 (rescuenet_seg_v1)']['genel'], REF['v2 (navitas_seg_v2)']['genel'])
    logla(f">>> En iyi öncekine göre kazanç: +{g-onceki:.3f}")
logla("\n✓ 2.2 TAMAM")

## 2.3 — Özet + zip + indir
Bu zip'i repo köküne koy → Claude DDR v4.3'e işler.

In [ ]:
import shutil
ozet = f"""# Navitas v3 Test — Özet
Tarih: {datetime.datetime.now().isoformat()}
Model SHA-256: {sha}
Ortam: {ortam.get('gpu')} · ultralytics {ortam.get('ultralytics')}

## Kıyas (aynı 294 RescueNet val)
| Model | mask mAP50 | traversable | blocked |
|---|---|---|---|
""" + "\n".join(f"| {a} | {d['genel']} | {d['traversable']} | {d['blocked']} |" for a,d in tablo.items()) + f"""

HEDEF ≥ 0,60 → v3 = {g} → {'GEÇTİ' if g>=0.60 else 'GEÇMEDİ'}

Dosyalar: konsol.log · ortam.json · v3_model_kimlik.json · v3_val_sonuclari.json ·
kiyas.json · v3_val/ (confusion, PR eğrileri, örnek kareler)
"""
open(f"{CALISMA}/00_OZET.md","w").write(ozet)
logla(ozet)
testzip = f"/content/navitas_v3_test_{datetime.datetime.now():%Y%m%d_%H%M}"
shutil.make_archive(testzip, "zip", CALISMA)
logla("Test zip:", testzip + ".zip")
from google.colab import files
files.download(testzip + ".zip")
logla("\n✓ 2.3 TAMAM — bitti! İki zip'i (weights + test) repoya koy, Claude'a haber ver.")

---
**Bitince:** `navitas_v3_test_*.zip` (ve mümkünse `navitas_v3_weights_*.zip`) repo köküne
koy, Claude'a haber ver → sonuçlar doğrulanıp **DDR v4.3**'e işlenecek, v3 modeli sahaya alınacak.

**Hedefi geçemediyse (v3 < 0,60, düşük ihtimal):** `--model yolov8m-seg.pt` ile 1.6'yı
tekrar çalıştır (daha büyük kapasite), veya blocked oversampling `--copies 3` yap.
Panik yok: v1 (0,619) zaten sahada kullanılabilir bir güvenlik ağı.